In [33]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper tables
PeriodInfo_df = pd.read_parquet(zip_folder / "statistics_all.parquet")

In [ ]:
# Now, we only focus on aces, so we filter our table and drop duplicates
PeriodInfo_df = PeriodInfo_df[(PeriodInfo_df["statistic_name"]=="aces") &
                              (PeriodInfo_df["period"]=="ALL")]
PeriodInfo_df.drop(columns=["date"],inplace=True)
PeriodInfo_df.drop_duplicates(inplace=True)

# Convert number of aces by each side to enable calculating total aces
PeriodInfo_df["home_stat"] = PeriodInfo_df["home_stat"].astype("Float64")
PeriodInfo_df["away_stat"] = PeriodInfo_df["away_stat"].astype("Float64")
PeriodInfo_df["total_aces"] = PeriodInfo_df["home_stat"] + PeriodInfo_df["away_stat"]

# Create clean data with only needed columns
PeriodInfo_clean = PeriodInfo_df[["match_id","home_stat","away_stat","total_aces"]]

In [ ]:
# Get mean with 2 approaches:
# 1: Do not omit outliers because the data is valid! (Based on Google search)
PeriodInfo_clean.describe()

In [ ]:
# 2: Omit outliers using IQR value
# Delete outliers of periods

Q1 = PeriodInfo_clean["total_aces"].quantile(0.25)
Q3 = PeriodInfo_clean["total_aces"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = PeriodInfo_clean[PeriodInfo_clean["total_aces"].isna() |
    ((PeriodInfo_clean["total_aces"] >= lower) &
     ( PeriodInfo_clean["total_aces"]<= upper))
]

df.describe()
